# 第13章 光谱数据、谱线与红移

这个 notebook 用一个极小的教学光谱表，演示波长轴、归一化流量、谱线结构和最小红移估计。

## 学习目标

- 理解波长与流量这两条基本轴
- 对比连续谱、吸收线和发射线
- 做一个最小红移试算
- 理解归一化和重采样为什么重要

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import defaultdict
from pathlib import Path

DATA_PATH = Path('../../data/small/spectral_wavelength_redshift_demo.csv')

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

spectra = defaultdict(list)
for row in rows:
    spectra[row['sample_id']].append(
        {
            'wavelength_rest_angstrom': float(row['wavelength_rest_angstrom']),
            'flux_norm': float(row['flux_norm']),
            'split': row['split'],
            'object_class': row['object_class'],
        }
    )

for sample_id in spectra:
    spectra[sample_id].sort(key=lambda item: item['wavelength_rest_angstrom'])

print(f'Loaded {len(rows)} wavelength rows from {DATA_PATH.name}')
print('Samples:', sorted(spectra))
print('Python version:', platform.python_version())


In [ ]:
for sample_id in sorted(spectra):
    wavelengths = [item['wavelength_rest_angstrom'] for item in spectra[sample_id]]
    fluxes = [item['flux_norm'] for item in spectra[sample_id]]
    print(sample_id, spectra[sample_id][0]['object_class'])
    print('  wavelength range:', (min(wavelengths), max(wavelengths)))
    print('  flux range:', (round(min(fluxes), 3), round(max(fluxes), 3)))


In [ ]:
def line_summary(sample):
    flux_by_wave = {item['wavelength_rest_angstrom']: item['flux_norm'] for item in sample}
    balmer_depth = 1.0 - flux_by_wave[4861.0]
    halpha_depth = 1.0 - flux_by_wave[6563.0]
    blue_red_slope = flux_by_wave[3900.0] - flux_by_wave[6563.0]
    return {
        'balmer_depth': round(balmer_depth, 3),
        'halpha_depth': round(halpha_depth, 3),
        'blue_red_slope': round(blue_red_slope, 3),
    }

for sample_id in ['B1', 'G1', 'E1']:
    print(sample_id, line_summary(spectra[sample_id]))


In [ ]:
rest_halpha = 6563.0
observed_halpha = 6694.0
redshift = (observed_halpha - rest_halpha) / rest_halpha
print('Minimal redshift example from H-alpha:')
print('  z =', round(redshift, 5))


In [ ]:
speed_km_s = 299792.458 * redshift
print('Small-redshift velocity approximation:')
print('  v ~=', round(speed_km_s, 1), 'km/s')


In [ ]:
def equivalent_width_proxy(sample, line_center):
    points = {item['wavelength_rest_angstrom']: item['flux_norm'] for item in sample}
    wavelengths = sorted(points)
    index = wavelengths.index(line_center)
    if index == 0:
        right_wave = wavelengths[index + 1]
        next_wave = wavelengths[index + 2]
        continuum = 0.5 * (points[right_wave] + points[next_wave])
        delta_lambda = next_wave - line_center
    elif index == len(wavelengths) - 1:
        left_wave = wavelengths[index - 1]
        prev_wave = wavelengths[index - 2]
        continuum = 0.5 * (points[left_wave] + points[prev_wave])
        delta_lambda = line_center - prev_wave
    else:
        left_wave = wavelengths[index - 1]
        right_wave = wavelengths[index + 1]
        continuum = 0.5 * (points[left_wave] + points[right_wave])
        delta_lambda = right_wave - left_wave
    return (1.0 - points[line_center] / continuum) * delta_lambda

for sample_id in ['B1', 'G1', 'E1']:
    balmer_ew = equivalent_width_proxy(spectra[sample_id], 4861.0)
    halpha_ew = equivalent_width_proxy(spectra[sample_id], 6563.0)
    print(sample_id, 'proxy EW near Hbeta/Halpha =', round(balmer_ew, 3), round(halpha_ew, 3))


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print('matplotlib unavailable; skipped spectrum plot:', exc)
else:
    fig, ax = plt.subplots(figsize=(6, 4))
    for sample_id, color in [('B1', 'tab:blue'), ('G1', 'tab:orange'), ('E1', 'tab:green')]:
        waves = [item['wavelength_rest_angstrom'] for item in spectra[sample_id]]
        fluxes = [item['flux_norm'] for item in spectra[sample_id]]
        ax.plot(waves, fluxes, marker='o', label=sample_id, color=color)
    ax.set_title('Teaching Spectra Demo')
    ax.set_xlabel('Wavelength [Angstrom]')
    ax.set_ylabel('Normalized Flux')
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    print('Displayed teaching spectra.')


## 解释

这个最小示例说明了光谱分析最重要的骨架：波长轴决定物理顺序，归一化流量决定结构对比，谱线深度和峰值帮助我们区分吸收与发射，而红移公式则把谱线位移转化成可解释的物理量。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'sample_ids': sorted(spectra),
    'B1_summary': line_summary(spectra['B1']),
    'G1_summary': line_summary(spectra['G1']),
    'E1_summary': line_summary(spectra['E1']),
    'demo_redshift': round(redshift, 5),
    'demo_velocity_km_s': round(speed_km_s, 1),
    'python_version': platform.python_version(),
}

print('Spectroscopy snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
